# TTS Dataset Publish

Reads an identity manifest (scored, per [`docs/manifest.md`](../docs/manifest.md))
and, optionally, a selection file, and builds a Hugging Face dataset folder:
a `wavs/` directory, an LJSpeech-style `metadata.csv`, the full `manifest.jsonl`
with a `selected` flag on every row, and a generated `README.md` dataset card.
It then optionally pushes the folder to the Hugging Face Hub.

This is step 9, Publish, of the [TTS dataset pipeline](../docs/pipeline.md). It
runs alone: give it a manifest path and it does not need any other notebook to
have run in the same session. With `MANIFEST_PATH` left empty, it builds one
from a few real `edge-tts` clips in its own fresh directory, so the notebook
has something to publish on its own; a `MANIFEST_PATH` that does not exist is
an error, never a silent switch to the demo.

> This notebook was written with AI assistance (Claude, via Claude Code) and has not been reviewed by a human. Verify outputs before relying on them.

> This project was funded through the NGI0 Commons Fund, a fund established by NLnet with financial support from the European Commission's Next Generation Internet programme, under the aegis of DG Communications Networks, Content and Technology under grant agreement No 101135429. Additional funding is made available by the Swiss State Secretariat for Education, Research and Innovation (SERI).

| | |
|---|---|
| **Inputs** | `MANIFEST_PATH`: a JSON Lines manifest (docs/manifest.md), or empty for the built-in demo. `SELECTION_PATH`: optional JSON Lines file of kept `id` values; absent, every row is published. |
| **Outputs** | `OUTPUT_DIR/wavs/<id>.wav`, `OUTPUT_DIR/metadata.csv`, `OUTPUT_DIR/manifest.jsonl` (with its `.meta.json` sidecar), `OUTPUT_DIR/README.md`; optionally a push to the Hugging Face Hub (`HF_PUSH=1`). |
| **Runtime** | SMOKE / the built-in demo: well under a minute on CPU (N_SAMPLES short edge-tts clips plus one that always fails the demo selection rule). A real manifest publishes at disk speed: hardlink or copy per row, no model inference. |
| **Hardware** | CPU only. No GPU is used or needed. |
| **Cost** | No paid compute. The only network calls are the optional Hub push and, in the demo path, a few free Edge TTS requests. |

CI: executed automatically by this repository's execute job on every push to `dev`/`master` and every pull request (`ngi_deliverable.skip_execute=false`, demo path only, no large download); the lint job separately validates notebook structure but does not parse or compile code cells.


In [ ]:
# Parameters
SMOKE = False   # True: run the built-in demo (a few short edge-tts clips), CPU, no large download
N_SAMPLES = 3   # demo clip count when SMOKE is True (the demo's 4th, cer-failing clip is always kept, to demonstrate the selection rule)


---
## 0 · Configuration

All parameters are set as environment variables via `os.environ.setdefault()`.
Override any of them from your shell, a `.env` file, or `%env` magic **before**
running the config cell.


In [ ]:
import os


def _default_base_dir(name):
    """Somewhere writable that survives the session on the current platform."""
    if os.path.isdir("/kaggle"):
        return f"/kaggle/working/{name}"
    if os.path.isdir("/content"):
        return f"/content/{name}"
    return f"./{name}"


DEMO_DIR = _default_base_dir("tts_publish_demo")

# ──────────────────────────────────────────────
# INPUT  (identity manifest and optional selection, docs/manifest.md)
#   Leave MANIFEST_PATH empty (the default) and Stage 1 below builds one from
#   three real edge-tts clips, in its own fresh directory under OUTPUT_DIR, so
#   the notebook has something to publish on its own. Set MANIFEST_PATH to a
#   real manifest to publish real data; a path that does not exist is an
#   error, never a silent switch to the demo.
# ──────────────────────────────────────────────
os.environ.setdefault("MANIFEST_PATH",   "")   # empty = the built-in demo
os.environ.setdefault("SELECTION_PATH",  "")   # JSON Lines of kept ids; empty = every row is published

# ──────────────────────────────────────────────
# OUTPUT  (the Hugging Face dataset folder)
# ──────────────────────────────────────────────
os.environ.setdefault("OUTPUT_DIR",      os.path.join(DEMO_DIR, "dataset"))

# ──────────────────────────────────────────────
# HUGGING FACE PUSH  (off by default)
# ──────────────────────────────────────────────
os.environ.setdefault("HF_PUSH",         "0")   # 1 = push OUTPUT_DIR to the Hub
os.environ.setdefault("HF_REPO",         "")    # e.g. TigreGotico/tts-train-synthetic-miro_pt-PT
os.environ.setdefault("HF_TOKEN",        "")    # write token (or use huggingface-cli login)


def _env(key, default=""):
    return os.environ.get(key, default)


def _flag(key):
    return _env(key, "0").strip().lower() in ("1", "true", "yes")


print("Configuration loaded.")
print(f"  Manifest    : {_env('MANIFEST_PATH') or '(empty — the built-in demo will be used)'}")
print(f"  Selection   : {_env('SELECTION_PATH') or '(none — every row is published)'}")
print(f"  Output dir  : {_env('OUTPUT_DIR')}")
print(f"  HF push     : {'yes -> ' + _env('HF_REPO') if _flag('HF_PUSH') else 'no'}")


---
## 1 · Manifest and selection

When `MANIFEST_PATH` does not exist, this stage builds one: three short
`edge-tts` clips with made-up scores, and a fourth clip whose made-up
character error rate (`cer`) is deliberately over a `0.10` bar, together with
a selection file that keeps the first three and drops the fourth. This is the
notebook's self-contained demo path — it needs no manifest to be handed to it,
only outbound access to the `edge-tts` service.

| Env var | Meaning |
|---|---|
| `MANIFEST_PATH` | The identity manifest to publish from. |
| `SELECTION_PATH` | Optional selection file. Empty means every row is published. |


In [ ]:
_manifest_path_cfg = _env("MANIFEST_PATH").strip()
demo_needed = _manifest_path_cfg == ""

if not demo_needed and not os.path.exists(_manifest_path_cfg):
    raise FileNotFoundError(
        f"MANIFEST_PATH={_manifest_path_cfg!r} does not exist. Set it to a real "
        f"manifest (docs/manifest.md), or leave it empty to run the built-in demo."
    )

if demo_needed:
    # edge-tts needs outbound internet access; only pulled in for the demo.
    %pip install --quiet --pre "edge-tts>=7.2.8" "nest_asyncio>=1.6.0" "soundfile>=0.14.0"
else:
    print(f"MANIFEST_PATH={_manifest_path_cfg} exists — the built-in demo is skipped.")


In [ ]:
import asyncio
import json
import subprocess
from pathlib import Path

if demo_needed:
    import nest_asyncio
    nest_asyncio.apply()

    # A fresh, self-owned scratch directory under OUTPUT_DIR: fixed filenames,
    # always safely overwritable, never derived from a user-supplied path, so
    # this stage never needs to remove a directory it did not create itself.
    demo_src_dir = Path(_env("OUTPUT_DIR")) / "_demo_source"
    demo_src_dir.mkdir(parents=True, exist_ok=True)
    manifest_path = demo_src_dir / "manifest.jsonl"
    os.environ["MANIFEST_PATH"] = str(manifest_path)

    # One clip per call, never padded (see docs/pipeline.md): each row is its
    # own edge-tts request and its own ffmpeg conversion. The last clip is
    # always kept: it is made to FAIL the selection rule below (a cer over the
    # 0.10 bar), so the demo always has one clip that the example selection
    # drops. N_SAMPLES controls how many of the passing clips run.
    ALL_DEMO_CLIPS = [
        {"id": "clip_0001", "text": "Hey Mycroft, what is the weather like today in Lisbon?", "cer": 0.02, "utmos": 3.8},
        {"id": "clip_0002", "text": "Set a timer for ten minutes.", "cer": 0.05, "utmos": 3.6},
        {"id": "clip_0003", "text": "Play some jazz music in the kitchen.", "cer": 0.01, "utmos": 3.9},
        {"id": "clip_0004", "text": "Turn off the lights in the living room.", "cer": 0.34, "utmos": 3.7},
    ]
    _n_pass = max(1, min(int(N_SAMPLES), len(ALL_DEMO_CLIPS) - 1))
    DEMO_CLIPS = ALL_DEMO_CLIPS[:_n_pass] + ALL_DEMO_CLIPS[-1:]
    DEMO_VOICE = "en-US-JennyNeural"

    async def _synth(text, voice, out_mp3):
        import edge_tts
        await edge_tts.Communicate(text, voice).save(str(out_mp3))

    import importlib.metadata

    import soundfile as sf

    def _edge_tts_version():
        try:
            return importlib.metadata.version("edge-tts")
        except importlib.metadata.PackageNotFoundError:
            return None

    demo_rows = []
    for clip in DEMO_CLIPS:
        mp3_path = demo_src_dir / f"{clip['id']}.mp3"
        wav_path = demo_src_dir / f"{clip['id']}.wav"
        asyncio.run(_synth(clip["text"], DEMO_VOICE, mp3_path))
        subprocess.run(
            ["ffmpeg", "-y", "-loglevel", "error", "-i", str(mp3_path),
             "-ar", "24000", "-ac", "1", str(wav_path)],
            check=True,
        )
        mp3_path.unlink()
        info = sf.info(str(wav_path))
        demo_rows.append({
            "id": clip["id"],
            "text": clip["text"],
            "lang": "en-US",
            "audio": wav_path.name,
            "duration_s": round(info.frames / info.samplerate, 3),
            "sample_rate": info.samplerate,
            # A plain donor-TTS clip, never converted: no "identity" is set,
            # since this demo carries no real person's voice. This demo calls
            # the edge_tts library directly, not the OVOS plugin, so the source
            # names the library actually used, not the plugin.
            "source": {"kind": "tts", "plugin": "edge_tts", "voice": DEMO_VOICE, "version": _edge_tts_version()},
            # These "scores" are made up for the demo, never measured; the
            # sidecar below marks them and every row explicitly as such.
            "scores": {"utmos": clip["utmos"], "cer": clip["cer"]},
        })

    with open(manifest_path, "w", encoding="utf-8") as f:
        for row in demo_rows:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")

    DEMO_INSTRUMENT_NOTE = "demo value from the notebook's own built-in demo path, not produced by a real instrument"
    with open(str(manifest_path) + ".meta.json", "w", encoding="utf-8") as f:
        json.dump({
            "notebooks": ["tts_publish (demo)"],
            "demo": True,
            "instruments": {
                "utmos": {"demo": True, "note": DEMO_INSTRUMENT_NOTE},
                "cer": {"demo": True, "note": DEMO_INSTRUMENT_NOTE},
            },
        }, f, indent=2)

    kept_rows = [r for r in demo_rows if r["scores"]["cer"] <= 0.10]
    removed = len(demo_rows) - len(kept_rows)
    selection_path = demo_src_dir / "selection.jsonl"
    with open(selection_path, "w", encoding="utf-8") as f:
        for row in kept_rows:
            f.write(json.dumps(row["id"]) + "\n")
    with open(str(selection_path) + ".meta.json", "w", encoding="utf-8") as f:
        json.dump({"rules": [{"rule": {"score": "cer", "op": "<=", "value": 0.10}, "removed": removed}]}, f, indent=2)

    if not _env("SELECTION_PATH"):
        os.environ["SELECTION_PATH"] = str(selection_path)

    print(f"Demo manifest: {len(demo_rows)} rows, {len(kept_rows)} pass cer <= 0.10, {removed} fails it.")
else:
    print("Using the given manifest and selection as-is.")


---
## 2 · Build the dataset folder

Reads the manifest and the selection, then writes `wavs/`, `metadata.csv` and
the full `manifest.jsonl`. Nothing is deleted or moved from the manifest's own
audio: files are hardlinked into `wavs/` when they share a filesystem with
`OUTPUT_DIR`, copied otherwise. `wavs/` itself is only ever rebuilt if a
previous run of this notebook created it (a marker file inside it says so);
otherwise an existing `OUTPUT_DIR/wavs` stops the notebook rather than being
overwritten.

`tts_enhance` and `tts_voice_convert` already copy `source` from the parent
row onto the row they produce, so most manifests need no lookup. This stage
still walks `parent_id` back to the nearest ancestor that carries a `source`
(see [docs/manifest.md](../docs/manifest.md)), as a fallback for a manifest
built another way, so the card below can always report where the speech
originally came from.

| Env var | Meaning |
|---|---|
| `OUTPUT_DIR` | The Hugging Face dataset folder to build. |


In [ ]:
import json
import os
import shutil
from pathlib import Path


def load_jsonl(path):
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows


def load_selection(path):
    """A selection file is a JSON Lines file of kept id values (docs/manifest.md).
    Returns None when no selection file is given: every row is then selected."""
    if not path or not os.path.exists(path):
        return None
    kept = set()
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            obj = json.loads(line)
            if isinstance(obj, str):
                kept.add(obj)
            elif isinstance(obj, dict) and "id" in obj:
                kept.add(obj["id"])
            else:
                raise ValueError(f"Selection line is neither an id string nor an object with 'id': {line!r}")
    return kept


def load_sidecar(manifest_or_selection_path):
    """The .meta.json sidecar next to a manifest or selection file. An absent
    sidecar is reported as absent, not fabricated: returns {}."""
    sidecar_path = manifest_or_selection_path + ".meta.json"
    if os.path.exists(sidecar_path):
        with open(sidecar_path, "r", encoding="utf-8") as f:
            return json.load(f)
    return {}


def resolve_source(row, by_id):
    """Walk parent_id back to the ancestor row that carries a 'source'."""
    seen = set()
    r = row
    while r is not None:
        if "source" in r:
            return r["source"]
        pid = r.get("parent_id")
        if pid is None or pid in seen:
            return None
        seen.add(pid)
        r = by_id.get(pid)
    return None


def source_label(source):
    if source is None:
        return "unknown"
    if source.get("kind") == "tts":
        return f"donor TTS: {source.get('plugin')} / {source.get('voice')}"
    if source.get("kind") == "corpus":
        name = source.get("name", "unknown corpus")
        url = source.get("url")
        return f"{name} ({url})" if url else name
    return "unknown"


def copy_or_link(src, dst):
    """Hardlink when src and dst share a filesystem; copy otherwise."""
    try:
        os.link(src, dst)
    except OSError:
        shutil.copy2(src, dst)


def selected_ids(rows, selection):
    if selection is None:
        return {r["id"] for r in rows}
    return {r["id"] for r in rows if r["id"] in selection}


def build_dataset_folder(manifest_path, selection_path, output_dir):
    rows = load_jsonl(manifest_path)
    by_id = {r["id"]: r for r in rows}
    selection = load_selection(selection_path)
    kept = selected_ids(rows, selection)

    manifest_dir = Path(manifest_path).parent
    wavs_dir = Path(output_dir) / "wavs"
    wavs_marker = wavs_dir / ".created_by_tts_publish"
    if wavs_dir.exists():
        if not wavs_marker.exists():
            raise RuntimeError(
                f"{wavs_dir} already exists and was not created by tts_publish "
                f"(no {wavs_marker.name} marker inside it). Refusing to rebuild it: "
                f"point OUTPUT_DIR somewhere new, or remove that directory yourself "
                f"if you are sure it holds nothing you need."
            )
        # Verified ours (the marker says so): safe to rebuild from scratch.
        shutil.rmtree(wavs_dir)
    wavs_dir.mkdir(parents=True)
    wavs_marker.write_text("written by tts_publish; safe for this notebook to rebuild\n", encoding="utf-8")

    meta_lines = []
    for row in sorted(rows, key=lambda r: r["id"]):
        if row["id"] not in kept:
            continue
        src = manifest_dir / row["audio"]
        dst = wavs_dir / f"{row['id']}.wav"
        copy_or_link(src, dst)
        meta_lines.append(f"{row['id']}|{row['text']}")

    with open(Path(output_dir) / "metadata.csv", "w", encoding="utf-8") as f:
        f.write("\n".join(meta_lines) + ("\n" if meta_lines else ""))

    out_manifest_path = Path(output_dir) / "manifest.jsonl"
    with open(out_manifest_path, "w", encoding="utf-8") as f:
        for row in rows:
            out_row = dict(row)
            out_row["selected"] = row["id"] in kept
            f.write(json.dumps(out_row, ensure_ascii=False) + "\n")

    manifest_sidecar = load_sidecar(manifest_path)
    selection_sidecar = load_sidecar(selection_path) if selection_path else {}
    out_sidecar = {
        "notebooks": manifest_sidecar.get("notebooks", []) + ["tts_publish"],
        "demo": manifest_sidecar.get("demo", False),
        "instruments": manifest_sidecar.get("instruments", {}),
        "selection": (
            {"path": os.path.basename(selection_path), **selection_sidecar}
            if selection_path and selection is not None
            else {"applied": False, "note": "no selection file was given; every row is selected"}
        ),
    }
    with open(str(out_manifest_path) + ".meta.json", "w", encoding="utf-8") as f:
        json.dump(out_sidecar, f, ensure_ascii=False, indent=2)

    return rows, by_id, kept, out_manifest_path


PUBLISH_ROWS, PUBLISH_BY_ID, PUBLISH_KEPT, PUBLISH_MANIFEST_OUT = build_dataset_folder(
    _env("MANIFEST_PATH"), _env("SELECTION_PATH"), _env("OUTPUT_DIR")
)
print(f"{len(PUBLISH_ROWS)} rows in the manifest, {len(PUBLISH_KEPT)} selected and copied into wavs/.")


---
## 3 · Generate the dataset card

The card is built from the selected rows and the sidecars, never typed by
hand: language and variety, voice identity, source, clip count and hours,
sample rate, the scores present with their instrument and version, and the
selection rules applied. The ownership and licence block matches the rest of
the collection.


In [ ]:
OWNERSHIP_BLOCK_REAL = """## Ownership and licensing

{who_line} The voice{plural}, this
dataset, and any model trained on it belong to TigreGotico Lda
(https://tigregotico.pt).

This dataset is licensed under Creative Commons
Attribution-NonCommercial-NoDerivatives 4.0 (CC BY-NC-ND 4.0). You may use
and share this dataset for non-commercial purposes only. Give attribution to
TigreGotico Lda. Do not modify, remix, or build derivative datasets or voices
from this data.

For commercial use, for derivative datasets, or for any other license of the
{who} voice identity, contact TigreGotico Lda."""

OWNERSHIP_BLOCK_GENERIC = """## Ownership and licensing

This dataset carries no voice-identity claim: {reason}. It belongs to
TigreGotico Lda (https://tigregotico.pt) for OpenVoiceOS.

This dataset is licensed under Creative Commons
Attribution-NonCommercial-NoDerivatives 4.0 (CC BY-NC-ND 4.0). You may use
and share this dataset for non-commercial purposes only. Give attribution to
TigreGotico Lda."""


def _format_rule(rule):
    """A selection rule: either the plain string this notebook's own demo
    writes, or the {"score"|"field": ..., "op": ..., "value": ...} object
    tts_select writes (docs/manifest.md). Never the raw dict repr."""
    if isinstance(rule, dict):
        key = rule.get("score", rule.get("field", "?"))
        return f"{key} {rule.get('op', '?')} {rule.get('value', '?')}"
    return str(rule)


def build_card(repo_name, rows, by_id, kept, sidecar):
    selected_rows = [r for r in rows if r["id"] in kept]
    is_demo = bool(sidecar.get("demo"))

    langs = sorted({r.get("lang", "unknown") for r in selected_rows})
    identities = sorted({r.get("identity") for r in selected_rows if r.get("identity")})
    if len(identities) == 1:
        identity = identities[0]
    elif identities:
        identity = "mixed: " + ", ".join(identities)
    else:
        identity = "none recorded (no voice conversion in this manifest)"

    srs = sorted({r.get("sample_rate") for r in selected_rows if r.get("sample_rate")})
    sample_rate = str(srs[0]) if len(srs) == 1 else (", ".join(str(s) for s in srs) if srs else "not recorded")

    total_hours = sum(r.get("duration_s", 0) for r in selected_rows) / 3600.0

    sources = {}
    for row in selected_rows:
        label = source_label(resolve_source(row, by_id))
        sources[label] = sources.get(label, 0) + 1

    score_keys = sorted({k for r in selected_rows for k in r.get("scores", {}).keys()})
    instruments = sidecar.get("instruments", {})
    selection_info = sidecar.get("selection", {})

    lines = []
    lines.append("---")
    lines.append("language:")
    for lg in langs:
        lines.append(f"- {lg.split('-')[0]}")
    lines.append("license: cc-by-nc-nd-4.0")
    lines.append("task_categories:")
    lines.append("- text-to-speech")
    lines.append("tags:")
    lines.append("- tts")
    lines.append("- speech")
    lines.append("---")
    lines.append("")
    lines.append(f"# {repo_name}")
    lines.append("")
    lines.append(
        f"This is a single-speaker synthetic speech dataset for **{', '.join(langs)}**. "
        f"It carries the **{identity}** voice identity. "
        f"The dataset contains {len(selected_rows)} recordings at {sample_rate} Hz, "
        f"{total_hours:.4f} hours of audio, with a `metadata.csv` transcript file "
        f"and one WAV file per line."
        + (" This dataset is the notebook's own built-in demo." if is_demo else "")
    )
    lines.append("")
    lines.append("## Source")
    lines.append("")
    for label, count in sorted(sources.items()):
        lines.append(f"- {label}: {count} clips")
    lines.append("")
    lines.append("## Scores")
    lines.append("")
    if score_keys:
        for key in score_keys:
            if is_demo:
                lines.append(f"- `{key}`: demo value from the notebook's own built-in demo path, not produced by a real instrument")
                continue
            instr = instruments.get(key)
            if instr:
                model = f", model {instr['model']}" if instr.get("model") else ""
                lines.append(f"- `{key}`: {instr.get('package', 'unknown package')} {instr.get('version', 'unknown version')}{model}")
            else:
                lines.append(f"- `{key}`: instrument not recorded in the manifest sidecar")
    else:
        lines.append("No scores are recorded on the selected rows.")
    lines.append("")
    lines.append("## Selection")
    lines.append("")
    if selection_info.get("applied") is False:
        lines.append(selection_info.get("note", "No selection file was given; every row is selected."))
    else:
        rules = selection_info.get("rules", [])
        if rules:
            for rule in rules:
                lines.append(f"- `{_format_rule(rule.get('rule'))}` removed {rule.get('removed')} rows")
        else:
            lines.append("A selection file was applied; its sidecar records no rules.")
    lines.append("")
    if is_demo:
        lines.append(OWNERSHIP_BLOCK_GENERIC.format(
            reason="it is the notebook's own built-in demo, using a synthetic donor voice with no voice conversion"
        ))
    elif identities:
        who = " and ".join(identities)
        if len(identities) == 1:
            who_line, plural = f"{who} is the recorded voice of a real person.", ""
        else:
            who_line, plural = f"{who} are the recorded voices of real people.", "s"
        lines.append(OWNERSHIP_BLOCK_REAL.format(who_line=who_line, plural=plural, who=who))
    else:
        lines.append(OWNERSHIP_BLOCK_GENERIC.format(
            reason="no row in this manifest went through voice conversion"
        ))
    lines.append("")
    return "\n".join(lines)


PUBLISH_SIDECAR = load_sidecar(str(PUBLISH_MANIFEST_OUT))
repo_name = _env("HF_REPO").split("/")[-1] or Path(_env("OUTPUT_DIR")).name
PUBLISH_CARD = build_card(repo_name, PUBLISH_ROWS, PUBLISH_BY_ID, PUBLISH_KEPT, PUBLISH_SIDECAR)
(Path(_env("OUTPUT_DIR")) / "README.md").write_text(PUBLISH_CARD, encoding="utf-8")
print(PUBLISH_CARD)


---
## 4 · Push to Hugging Face (optional)

Off by default. With `HF_PUSH=1`, the folder is pushed into `HF_REPO` (for
example `TigreGotico/tts-train-synthetic-miro_pt-PT`), one commit through
`upload_folder` for a folder of up to 10,000 files, `upload_large_folder`
above that. After the push, the repo's file list is read back and the
pushed WAV count is checked against the number of selected rows.

| Env var | Meaning |
|---|---|
| `HF_PUSH` | `1` to push `OUTPUT_DIR` to the Hub. |
| `HF_REPO` | Target dataset repo, for example `TigreGotico/tts-train-synthetic-miro_pt-PT`. |
| `HF_TOKEN` | Write token (or use `huggingface-cli login`). |


In [ ]:
if _flag("HF_PUSH"):
    %pip install --quiet --pre "huggingface_hub>=0.36.0,<2.0"
else:
    print("HF_PUSH not enabled — skipping install.")


In [ ]:
# The marker this notebook writes into wavs/, and, on the demo path, the
# clip its own selection dropped, are local bookkeeping: never published.
PUSH_IGNORE_PATTERNS = ["*.created_by_tts_publish", "_demo_source/*"]

if not _flag("HF_PUSH"):
    n_wavs = len(list((Path(_env("OUTPUT_DIR")) / "wavs").glob("*.wav")))
    print("HF_PUSH=0 — not pushing. Would upload:")
    print(f"  repo       : {_env('HF_REPO') or '(HF_REPO not set)'}")
    print(f"  folder     : {_env('OUTPUT_DIR')}")
    print(f"  wavs       : {n_wavs}")
    print(f"  metadata   : metadata.csv, manifest.jsonl, manifest.jsonl.meta.json, README.md")
    print(f"  ignored    : {PUSH_IGNORE_PATTERNS}")
else:
    repo_id = _env("HF_REPO").strip()
    if not repo_id:
        raise RuntimeError(
            "HF_PUSH=1 but HF_REPO is empty. Set HF_REPO, e.g. "
            "TigreGotico/tts-train-synthetic-miro_pt-PT."
        )

    from huggingface_hub import HfApi, upload_large_folder

    api = HfApi(token=_env("HF_TOKEN") or None)
    api.create_repo(repo_id=repo_id, repo_type="dataset", exist_ok=True)

    output_dir = Path(_env("OUTPUT_DIR"))
    n_files = sum(1 for p in output_dir.rglob("*") if p.is_file())
    print(f"Pushing {n_files} files to {repo_id} ...")

    if n_files > 10000:
        # upload_large_folder batches the commit itself and takes no delete_patterns:
        # a selection that shrank above this size needs a manual prune of the
        # remote wavs/ before this push, or the count check below will fail.
        upload_large_folder(
            repo_id=repo_id, repo_type="dataset", folder_path=str(output_dir),
            ignore_patterns=PUSH_IGNORE_PATTERNS,
        )
    else:
        api.upload_folder(
            folder_path=str(output_dir),
            repo_id=repo_id,
            repo_type="dataset",
            commit_message="Publish TTS dataset",
            ignore_patterns=PUSH_IGNORE_PATTERNS,
            # wavs/ always mirrors this run's selection: a WAV a previous push
            # left on the Hub that this run no longer selects is deleted in
            # the same commit, so the count check below can hold.
            delete_patterns=["wavs/*"],
        )

    remote_files = api.list_repo_files(repo_id=repo_id, repo_type="dataset")
    remote_wavs = [f for f in remote_files if f.startswith("wavs/") and f.endswith(".wav")]
    expected = len(PUBLISH_KEPT)
    assert len(remote_wavs) == expected, (
        f"pushed {len(remote_wavs)} wavs, expected {expected} selected rows"
    )
    print(f"Verified: {len(remote_wavs)} wavs on the Hub match {expected} selected rows.")
    print(f"  https://huggingface.co/datasets/{repo_id}")


---
## 5 · Output summary


In [ ]:
output_dir = Path(_env("OUTPUT_DIR"))
n_wavs = len(list((output_dir / "wavs").glob("*.wav")))
meta_n = sum(1 for line in open(output_dir / "metadata.csv", encoding="utf-8") if line.strip())
manifest_n = sum(1 for line in open(output_dir / "manifest.jsonl", encoding="utf-8") if line.strip())

print("=" * 60)
print("  OUTPUT SUMMARY")
print("=" * 60)
print(f"  {'wavs/':20s} {n_wavs:>6} files")
print(f"  {'metadata.csv':20s} {meta_n:>6} rows")
print(f"  {'manifest.jsonl':20s} {manifest_n:>6} rows ({n_wavs} selected, {manifest_n - n_wavs} not)")
marker = "OK" if n_wavs == meta_n else "MISMATCH"
print(f"\n  wavs vs metadata.csv: {marker}")


---
Licensed under Apache-2.0.
